In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 03 safe lag 主版本

## 目标
固定当前最优且更可信的版本，作为本周的主版本。

## 当前最佳版本
- 特征：stores + oil + basic_date_features + lag_16 + lag_21 + lag_28
- 模型：CatBoostRegressor
- 验证方式：最近 365 天，最后 16 天作为验证集
- Validation RMSLE：0.447398

## 为什么保留这个版本
- baseline 提供了稳定起点
- short lag 虽然分数更低，但验证偏乐观
- safe lag 在更真实的设定下依然显著优于 baseline，因此更适合作为当前主版本

In [2]:
# ========================================
# Store Sales - 当前最佳主版本（safe lag）
# 说明：
# 1. 这是当前最适合继续学习和工作的版本
# 2. 当前最佳特征是：基础特征 + lag_16 + lag_21 + lag_28
# 3. 当前最可信验证分数大约是：0.447398
# 4. 这份代码不包含 holiday，也不包含 rolling，因为它们当前都不作为主版本
# ========================================

import pandas as pd
import numpy as np
from catboost import CatBoostRegressor

# ========================================
# 0. 读取数据
# ========================================
PATH = "/kaggle/input/competitions/store-sales-time-series-forecasting/"

train = pd.read_csv(PATH + "train.csv", parse_dates=["date"])
test = pd.read_csv(PATH + "test.csv", parse_dates=["date"])
stores = pd.read_csv(PATH + "stores.csv")
oil = pd.read_csv(PATH + "oil.csv", parse_dates=["date"])

# holidays / transactions 先读进来，当前主版本不直接使用
holidays = pd.read_csv(PATH + "holidays_events.csv", parse_dates=["date"])
transactions = pd.read_csv(PATH + "transactions.csv", parse_dates=["date"])

sample_submission = pd.read_csv(PATH + "sample_submission.csv")

print("train shape:", train.shape)
print("test shape:", test.shape)
print("stores shape:", stores.shape)
print("oil shape:", oil.shape)


# ========================================
# 1. 先搭“数据底座”
# ========================================
base_train = train.merge(stores, on="store_nbr", how="left")
base_test = test.merge(stores, on="store_nbr", how="left")

base_train = base_train.merge(oil, on="date", how="left")
base_test = base_test.merge(oil, on="date", how="left")

print("base_train shape:", base_train.shape)
print("base_test shape:", base_test.shape)


# ========================================
# 2. 处理 oil 缺失值
# ========================================
base_train = base_train.sort_values("date").copy()
base_test = base_test.sort_values("date").copy()

base_train["dcoilwtico"] = base_train["dcoilwtico"].ffill().bfill()
base_test["dcoilwtico"] = base_test["dcoilwtico"].ffill().bfill()

print("base_train oil missing:", base_train["dcoilwtico"].isna().sum())
print("base_test oil missing:", base_test["dcoilwtico"].isna().sum())


# ========================================
# 3. 复制一份工作表
# ========================================
train_df = base_train.copy()
test_df = base_test.copy()


# ========================================
# 4. 构造基础日期特征
# ========================================
for df in [train_df, test_df]:
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["dayofyear"] = df["date"].dt.dayofyear
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)


# ========================================
# 5. 定义基础特征列
# ========================================
cat_cols = ["store_nbr", "family", "city", "state", "type", "cluster"]

num_cols = [
    "onpromotion",
    "dcoilwtico",
    "year",
    "month",
    "day",
    "dayofweek",
    "dayofyear",
    "is_weekend"
]

feature_cols = cat_cols + num_cols
target_col = "sales"

for col in cat_cols:
    train_df[col] = train_df[col].astype(str)
    test_df[col] = test_df[col].astype(str)


# ========================================
# 6. 构造 safe lag 特征
#
# 为什么要先把 train 和 test 拼起来？
# 因为 test 没有真实 sales，
# 但 test 的 lag 可以从 train 末尾历史里“借过来”
#
# 为什么按 store_nbr + family 分组？
# 因为 lag 必须在“同一个门店、同一个商品家族”内部计算
# ========================================
lag_df = pd.concat([
    train_df[["id", "date", "store_nbr", "family", "sales"]].copy(),
    test_df[["id", "date", "store_nbr", "family"]].assign(sales=np.nan).copy()
], ignore_index=True)

lag_df = lag_df.sort_values(["store_nbr", "family", "date"]).copy()

lag_df["lag_16"] = lag_df.groupby(["store_nbr", "family"])["sales"].shift(16)
lag_df["lag_21"] = lag_df.groupby(["store_nbr", "family"])["sales"].shift(21)
lag_df["lag_28"] = lag_df.groupby(["store_nbr", "family"])["sales"].shift(28)
lag_df["rolling_mean_14_from_shift16"] = (
    lag_df.groupby(["store_nbr", "family"])["sales"]
    .transform(lambda x: x.shift(16).rolling(window=14, min_periods=14).mean())
)
# 把 lag 特征合回训练集和测试集
train_df_lag = train_df.merge(
    lag_df[["id", "lag_16", "lag_21", "lag_28", "rolling_mean_14_from_shift16"]],
    on="id",
    how="left"
)

test_df_lag = test_df.merge(
    lag_df[["id", "lag_16", "lag_21", "lag_28", "rolling_mean_14_from_shift16"]],
    on="id",
    how="left"
)

print("train lag missing:")
print(train_df_lag[["lag_16", "lag_21", "lag_28"]].isna().sum())

print("test lag missing:")
print(test_df_lag[["lag_16", "lag_21", "lag_28"]].isna().sum())


# ========================================
# 7. 填充 lag 缺失值
# ========================================
fill_cols = ["lag_16", "lag_21", "lag_28", "rolling_mean_14_from_shift16"]

for col in fill_cols:
    train_df_lag[col] = train_df_lag[col].fillna(0)
    test_df_lag[col] = test_df_lag[col].fillna(0)

feature_cols_rolling14 = feature_cols + [
    "lag_16", "lag_21", "lag_28", "rolling_mean_14_from_shift16"
]
# ========================================
# 8. 只保留最近 365 天数据
# ========================================
cutoff_date = train_df_lag["date"].max() - pd.Timedelta(days=365)
train_df_small = train_df_lag[train_df_lag["date"] >= cutoff_date].copy()
train_df_small = train_df_small.sort_values(["store_nbr", "family", "date"]).copy()


# ========================================
# 9. 按时间切分验证集（最后 16 天）
# ========================================
last_date = train_df_small["date"].max()
val_start_date = last_date - pd.Timedelta(days=15)

print("train max date:", last_date)
print("validation start date:", val_start_date)

train_part = train_df_small[train_df_small["date"] < val_start_date].copy()
valid_part = train_df_small[train_df_small["date"] >= val_start_date].copy()

print("train_part shape:", train_part.shape)
print("valid_part shape:", valid_part.shape)
print("train_part date range:", train_part["date"].min(), "to", train_part["date"].max())
print("valid_part date range:", valid_part["date"].min(), "to", valid_part["date"].max())


# ========================================
# 10. 构造训练输入 / 验证输入 / 测试输入
# ========================================
X_train = train_part[feature_cols_rolling14].copy()
X_valid = valid_part[feature_cols_rolling14].copy()
X_test = test_df_lag[feature_cols_rolling14].copy()

y_train = np.log1p(train_part[target_col].values)
y_valid = np.log1p(valid_part[target_col].values)

print("当前训练特征列：")
print(X_train.columns.tolist())


# ========================================
# 11. 训练 CatBoost 模型
# ========================================
model = CatBoostRegressor(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=50
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_valid, y_valid),
    use_best_model=True,
    early_stopping_rounds=50
)


# ========================================
# 12. 在验证集上评估
# ========================================
valid_pred_log = model.predict(X_valid)
valid_pred = np.expm1(valid_pred_log)
valid_pred = np.clip(valid_pred, 0, None)

valid_true = valid_part[target_col].values

def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))

score = rmsle(valid_true, valid_pred)
print("Validation RMSLE:", score)


# ========================================
# 13. 生成 submission 文件
# ========================================
test_pred_log = model.predict(X_test)
test_pred = np.expm1(test_pred_log)
test_pred = np.clip(test_pred, 0, None)

submission = pd.DataFrame({
    "id": test_df["id"],
    "sales": test_pred
})

print(submission.head())

submission.to_csv("submission_safe_lag_plus_rolling14.csv", index=False)
print("submission_safe_lag_plus_rolling14.csv saved!")

train shape: (3000888, 6)
test shape: (28512, 5)
stores shape: (54, 5)
oil shape: (1218, 2)
base_train shape: (3000888, 11)
base_test shape: (28512, 10)
base_train oil missing: 0
base_test oil missing: 0
train lag missing:
lag_16    28512
lag_21    37422
lag_28    49896
dtype: int64
test lag missing:
lag_16    0
lag_21    0
lag_28    0
dtype: int64
train max date: 2017-08-15 00:00:00
validation start date: 2017-07-31 00:00:00
train_part shape: (621918, 21)
valid_part shape: (28512, 21)
train_part date range: 2016-08-15 00:00:00 to 2017-07-30 00:00:00
valid_part date range: 2017-07-31 00:00:00 to 2017-08-15 00:00:00
当前训练特征列：
['store_nbr', 'family', 'city', 'state', 'type', 'cluster', 'onpromotion', 'dcoilwtico', 'year', 'month', 'day', 'dayofweek', 'dayofyear', 'is_weekend', 'lag_16', 'lag_21', 'lag_28', 'rolling_mean_14_from_shift16']
0:	learn: 2.4580127	test: 2.4097110	best: 2.4097110 (0)	total: 488ms	remaining: 2m 25s
50:	learn: 0.5809415	test: 0.5073887	best: 0.5073887 (50)	total: 1

safe_lag_16_21_28_catboost_recent365
features_used: stores + oil + basic_date_features + lag_16 + lag_21 + lag_28
validation_split: last_16_days
model: CatBoostRegressor
params: iterations=300, depth=6, lr=0.05
score: 0.442885
notes: rerun of pure safe lag main version; current best trusted baseline if setup is unchanged
当前主版本为 safe lag（lag_16 / lag_21 / lag_28）+ basic date features + stores + oil，当前最可信 validation RMSLE = 0.442885。

之所以把它作为当前主版本，不只是因为分数更好，也因为它比 short lag（lag_7 / lag_14）更符合当前最后 16 天验证设定下的“相对安全比较”原则。

当前主版本为 safe lag（lag_16 / lag_21 / lag_28）+ basic date features + stores + oil，当前最可信 Validation RMSLE = 0.442885。

之所以把它作为当前主版本，不只是因为它目前分数更好，也因为它比 short lag（lag_7 / lag_14）更符合当前“最后 16 天做 validation”的相对安全比较原则。

因此，当前 rolling 方向以及后续新特征，默认都应以 0.442885 作为新的主比较基准。
本周 rolling 方向最后审判基准：0.442885

构造逻辑：
按 store_nbr + family 分组，按 date 排序，对 sales 先 shift(16)，再 rolling(14).mean()，得到 rolling_mean_14_from_shift16。目的是在最后 16 天 validation 设定下，尽量避免直接读到 validation 窗口内部真实销量，同时测试更长窗口的 rolling 均值是否比 rolling_mean_7 更有用。

leakage 说明：
该 rolling 特征不是直接对当前及最近几天的 sales 做均值，而是先 shift(16)，再 rolling(14)。因此在当前最后 16 天作为 validation 的设定下，构造时尽量只使用更早的历史销量，避免直接依赖 validation 窗口内部真实销量。它仍属于基于历史 target 的时序特征，但相比 short lag 或未安全偏移的 rolling，当前写法更适合做相对安全比较。

代码改动：
在 pure safe lag 主版本基础上，新增 rolling_mean_14_from_shift16；其他设定保持不变。

validation 分数：
（把你跑出来的分数填这里）

构造逻辑：
按 store_nbr + family 分组，按 date 排序，对 sales 先 shift(16)，再 rolling(14).mean()。

leakage 说明：
先 shift(16) 再 rolling(14)，尽量避免直接读到最后 16 天 validation 窗口内部真实销量。

初步备注：
与当前主版本 0.442885 比较，先记结果，不在周二提前判决 rolling 去留。

实验名：
safe_lag_plus_safe_rolling_mean_14_catboost_recent365

代码改动：
在 pure safe lag 主版本基础上，新增 rolling_mean_14_from_shift16；其余设定保持不变，包括 recent 365 days、last 16 days validation、CatBoost 参数不变。

validation 分数：
0.4529245221

构造逻辑：
按 store_nbr + family 分组，按 date 排序，对 sales 先 shift(16)，再 rolling(14).mean()，得到 rolling_mean_14_from_shift16。

leakage 说明：
该特征先 shift(16)，再 rolling(14)，在当前最后 16 天作为 validation 的设定下，尽量只使用更早的历史销量，避免直接依赖 validation 窗口内部真实销量，因此可作为相对安全的 rolling 比较版本。

初步备注：
该版本明显差于当前 pure safe lag 主版本 0.442885，也差于预期，说明把 rolling 窗口从 7 扩到 14 并没有带来提升。

三版比较：

- pure safe lag：0.442885
- safe lag + rolling_mean_7：0.450380
- safe lag + rolling_mean_14：0.452925

比较结论：

- 14 天 rolling 没有比 7 天 rolling 更好，反而更差。
- rolling_7 和 rolling_14 都没有超过当前主版本 pure safe lag。
- 因此，rolling 当前无效不太像只是窗口长度问题，更像是 rolling 方向在当前设定下没有提供有效增益。

当前判断：

- 当前主版本仍为 pure safe lag。
- 当前主比较基准仍为 0.442885。
- rolling 方向继续的必要性已经很弱，周四再做正式保留/不保留决策。